# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sayuj5/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import subprocess, os

# Clone repo if not already present
if not os.path.exists('/content/flyrank-internship-ml'):
    subprocess.run(['git', 'clone', 'https://github.com/sayuj5/flyrank-internship-ml.git'],
                   capture_output=True, text=True)
    print("Repo cloned successfully")
else:
    print("Repo already exists")

# Verify data file is there
print(os.path.exists('/content/flyrank-internship-ml/data/raw/content_refresh_anonymized.csv'))

Repo already exists
True


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: Scoring (regression)

This is a scoring task. The goal is to assign each article a continuous
engagement score based on predicted clicks_90d. Scoring is the right choice
because engagement exists on a spectrum — editors need to know how much a piece
is underperforming, not just whether it is. A classification approach (good/bad)
would discard the magnitude, which is exactly what's needed to prioritize a
queue of 30,000 articles. The output is a ranked list: articles at the bottom
get editorial attention first.

In [5]:
print("Task type: Scoring (regression)")
print("Output: continuous engagement score per article (predicted clicks_90d)")
print("Unit: one article = one row")

Task type: Scoring (regression)
Output: continuous engagement score per article (predicted clicks_90d)
Unit: one article = one row


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target (proxy): clicks_90d — total clicks over the past 90 days per article.

There is no direct "content quality" label in the data, so clicks_90d is used
as a behavioral proxy. It is an observed outcome reflecting whether readers
actually visited the article after seeing it in search results. It comes from
real user behavior, not an editorial rule.

A secondary proxy is low_engagement — a binary flag where 1 = clicks_90d at or
below the dataset median (1 click), capturing the large underperforming tail.
engagement_rate and ctr are supporting signals but are downstream of clicks,
so clicks_90d is the cleaner primary target.

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/flyrank-internship-ml/data/raw/content_refresh_anonymized.csv')

median_clicks = df['clicks_90d'].median()
df['low_engagement'] = (df['clicks_90d'] <= median_clicks).astype(int)

print(f"Target column: clicks_90d")
print(f"Median clicks (threshold): {median_clicks:.0f}")
print(f"Low engagement articles:  {df['low_engagement'].sum():,} / {len(df):,} ({df['low_engagement'].mean()*100:.1f}%)")
print(f"High engagement articles: {(df['low_engagement']==0).sum():,} / {len(df):,}")

Target column: clicks_90d
Median clicks (threshold): 1
Low engagement articles:  17,025 / 30,000 (56.8%)
High engagement articles: 12,975 / 30,000


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary metric: RMSE (Root Mean Squared Error) on predicted clicks_90d.
Secondary metric: AUC-ROC on the binary low_engagement flag.

"Good" means: RMSE lower than the naive baseline of always predicting the mean
(16.1 clicks). A 20%+ reduction over baseline RMSE would justify replacing
the current rule-based refresh queue. For the binary version, AUC > 0.70
indicates real discriminative power beyond random chance.

The metric ties directly to the action: editors only benefit if the score
correctly separates underperforming articles from healthy ones. A model that
just predicts the mean for every article wastes their time.

In [7]:
mean_clicks = df['clicks_90d'].mean()
baseline_rmse = np.sqrt(((df['clicks_90d'] - mean_clicks) ** 2).mean())

print(f"Mean clicks (baseline prediction): {mean_clicks:.2f}")
print(f"Baseline RMSE (always predict mean): {baseline_rmse:.2f}")
print(f"Target: beat RMSE < {baseline_rmse * 0.8:.2f} (20% improvement over baseline)")
print(f"\nAUC baseline (random classifier): 0.50")
print(f"Target AUC: > 0.70")

Mean clicks (baseline prediction): 16.10
Baseline RMSE (always predict mean): 75.08
Target: beat RMSE < 60.06 (20% improvement over baseline)

AUC baseline (random classifier): 0.50
Target AUC: > 0.70


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one article.

Each article is scored independently. Features include structural signals
(word_count, char_count), search signals (search_volume, avg_position, ctr),
freshness signals (content_age_days, days_since_last_update), and behavioral
signals (impressions_90d, engagement_rate, scroll_rate). The target column is
clicks_90d with a derived low_engagement binary flag.

In [8]:
display_cols = [
    'content_id', 'word_count', 'avg_position', 'search_volume',
    'content_age_days', 'days_since_last_update', 'impressions_90d',
    'engagement_rate', 'ctr', 'clicks_90d', 'low_engagement'
]

print(f"Unit of analysis: one article per row")
print(f"Total articles: {len(df):,}")
print(f"Features shown: {len(display_cols)-2} (excluding target columns)")
print()
df[display_cols].head(10)

Unit of analysis: one article per row
Total articles: 30,000
Features shown: 9 (excluding target columns)



,content_id,word_count,avg_position,search_volume,content_age_days,days_since_last_update,impressions_90d,engagement_rate,ctr,clicks_90d,low_engagement
0,content_304f48230142,3221.0,10.6,10.0,187,20,3803,5.88,0.76,29,0
1,content_a1fb4e703a9e,2481.0,20.3,90.0,445,25,15320,0.00,0.05,7,0
2,content_9aa793d4d895,3515.0,36.5,0.0,141,20,12581,0.00,0.09,11,0
3,content_331d6c4de07b,NaN,6.2,10.0,463,22,11751,1.28,0.49,58,0
4,content_d99b7a2d90ca,2803.0,44.0,0.0,263,14,19140,0.00,0.13,24,0
5,content_d4084a4bc775,3080.0,8.5,720.0,147,20,3970,0.00,0.03,1,1
6,content_9a34b442b552,3059.0,7.0,0.0,90,20,20,0.00,0.00,0,1
7,content_a63219c6e95a,NaN,21.2,590.0,445,22,1724,3.57,0.06,1,1
8,content_5e6c160719bc,3807.0,46.0,0.0,90,20,32574,5.88,0.09,29,0
9,content_c27558df2b0c,NaN,4.9,0.0,257,104,1240,0.00,0.16,2,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule might say: "flag any article with clicks_90d < 5 for refresh."
But this ignores context entirely — a low-click article on a low search_volume
topic may be performing at its ceiling, while a high search_volume article with
the same clicks is massively underperforming.

The pattern is too messy for an if-statement because engagement is shaped by
the interaction of avg_position × search_volume × content_age_days × word_count
× days_since_last_update simultaneously. No single threshold captures this.

ML learns the weight of each factor from 30,000 real articles, adapting to what
actually predicts clicks in this corpus. The output — a ranked engagement score
— directly feeds the refresh queue: editors sort by score, review the bottom
quartile, and decide which articles to rewrite, re-promote, or retire.

In [9]:
low_vol  = df[df['search_volume'] < 10]['clicks_90d'].mean()
high_vol = df[df['search_volume'] >= 100]['clicks_90d'].mean()

young = df[df['content_age_days'] < 150]['clicks_90d'].mean()
old   = df[df['content_age_days'] >= 400]['clicks_90d'].mean()

print("Why a single clicks threshold fails:")
print(f"  Avg clicks, low search volume  (< 10):   {low_vol:.1f}")
print(f"  Avg clicks, high search volume (>= 100): {high_vol:.1f}")
print(f"\n  Avg clicks, young content (< 150 days):  {young:.1f}")
print(f"  Avg clicks, old content   (>= 400 days): {old:.1f}")
print("\nSame click count means very different things depending on context.")
print("ML handles these interactions simultaneously; a fixed rule cannot.")

Why a single clicks threshold fails:
  Avg clicks, low search volume  (< 10):   18.6
  Avg clicks, high search volume (>= 100): 13.0

  Avg clicks, young content (< 150 days):  18.5
  Avg clicks, old content   (>= 400 days): 13.7

Same click count means very different things depending on context.
ML handles these interactions simultaneously; a fixed rule cannot.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.